# Session 4 Live - Reliable AI and Human Review

## Objective

A selected classifier is converted into a controlled AI decision engine. Every request must enter exactly one explicit state:

- `ACCEPT_AI_DECISION`
- `HUMAN_REVIEW_LOW_CONFIDENCE`
- `HUMAN_REVIEW_ANOMALY`
- `REJECT_INVALID_INPUT`

**Primary artifact:** a Reliable AI Decision Router supported by validation, explanation, calibration, threshold policy, input validation, anomaly review, and audit evidence.


## Bridge from Session 3 to Session 4

Session 3 was used to **inspect** the selected model:

1. class-balanced performance was checked;
2. the predicted class was identified;
3. global feature reliance was measured;
4. one-case sensitivity and SHAP contributions were examined;
5. explanation stability was checked.

Session 4 is used to **control** whether that model may act:

1. confidence is calibrated and measured;
2. an automatic-action threshold is selected from calibration data;
3. impossible requests are rejected before prediction;
4. valid but unfamiliar requests are deferred;
5. exactly one routing state is returned.

> **Core bridge:** Session 3 asked, “Why was this output produced?” Session 4 asks, “Should this output be accepted, reviewed, or refused?”


## Complete symbol map for Sessions 3 and 4

The same notation is used throughout the notebook. The table is intentionally placed before the equations so that no symbol appears without context.

| Area | Symbol | Meaning | Plain-language interpretation |
|---|---|---|---|
| General | $x$ | One input case | One region-month request supplied to the system |
| General | $X$ | Input dataset | All input cases used for fitting or evaluation |
| General | $x_i$ | Case $i$ | The $i$-th record |
| General | $x_j$ | Feature $j$ in a case | One measurement, such as `drought_index` |
| General | $i$ | Case index | Identifies one record |
| General | $j$ | Feature index | Identifies one input feature |
| General | $M$ | Number of features | Total model-input columns |
| General | $c$ | Class index | `LOW`, `MEDIUM`, or `HIGH` |
| General | $C$ | Number of classes | Here, $C=3$ |
| General | $n$ | Number of evaluated cases | Total records included in a metric |
| General | $y_i$ | True class of case $i$ | The observed label |
| General | $\widehat{y}_i$ | Predicted class of case $i$ | The class selected by the model |
| General | $\mathbb{1}[\cdot]$ | Indicator function | Equals $1$ when a condition is true and $0$ otherwise |
| Validation | $P_c$ | Precision for class $c$ | Of predictions assigned to class $c$, the correct share |
| Validation | $R_c$ | Recall for class $c$ | Of true class-$c$ cases, the detected share |
| Validation | $TP_c$ | True positives for class $c$ | Correct class-$c$ predictions |
| Validation | $FP_c$ | False positives for class $c$ | Other classes incorrectly predicted as class $c$ |
| Validation | $FN_c$ | False negatives for class $c$ | Class-$c$ cases missed by the model |
| Validation | $F_{1,c}$ | F1 score for class $c$ | Precision-recall balance for one class |
| Validation | $F_{1,\mathrm{macro}}$ | Macro F1 | Equal-weight average of class F1 scores |
| Prediction | $\widehat{p}_c(x)$ | Probability assigned to class $c$ | Model confidence for one class |
| Prediction | $c^*(x)$ | Predicted class | Class with the largest probability |
| Prediction | $\kappa(x)$ | Prediction confidence | Maximum class probability |
| Permutation | $S_0$ | Baseline evaluation score | Score before a feature is shuffled |
| Permutation | $S^{\mathrm{perm}}_{j,r}$ | Permuted score | Score after feature $j$ is shuffled in repetition $r$ |
| Permutation | $r$ | Repetition index | One shuffle experiment |
| Permutation | $R$ | Number of repetitions | Total shuffles per feature |
| Permutation | $I_j$ | Permutation importance | Average score decrease after feature $j$ is shuffled |
| Occlusion | $b_j$ | Reference value for feature $j$ | Usually the training median |
| Occlusion | $x_{j\leftarrow b_j}$ | Modified case | Case $x$ after feature $j$ is replaced by $b_j$ |
| Occlusion | $\Delta_{j,c^*}(x)$ | Local occlusion effect | Change in selected-class probability after replacement |
| SHAP | $F$ | Set of all features | Complete feature set |
| SHAP | $S$ | Feature coalition | A subset of features that does not contain feature $j$ |
| SHAP | $v_c(S)$ | Coalition value | Class-$c$ model output when coalition $S$ is known |
| SHAP | $\phi_{j,c}$ | SHAP value | Contribution assigned to feature $j$ for class $c$ |
| SHAP | $f_c(x)$ | Class-specific model output | Score being explained for class $c$ |
| SHAP | $\mathbb{E}[f_c(X)]$ | SHAP baseline | Reference class-$c$ output before this case is explained |
| Stability | $A,B$ | Two top-feature sets | Top factors obtained from two explanation backgrounds |
| Stability | $J(A,B)$ | Jaccard overlap | Shared factors divided by all unique factors |
| Calibration | $z_c(x)$ | Logit for class $c$ | Raw class score before softmax |
| Calibration | $z(x)$ | Logit vector | All class logits for case $x$ |
| Calibration | $T$ | Temperature | Positive scalar controlling probability sharpness |
| Calibration | $T^*$ | Selected temperature | Temperature minimizing calibration negative log-likelihood |
| Calibration | $\mathcal{L}_{\mathrm{NLL}}$ | Negative log-likelihood | Calibration loss used to fit $T$ |
| Calibration | $D_{\mathrm{cal}}$ | Calibration dataset | Data used to fit temperature and policy |
| Calibration | $y_{\mathrm{cal}}$ | Calibration labels | True labels from the calibration period |
| Calibration | $z_{\mathrm{cal}}$ | Calibration logits | Model logits on calibration cases |
| Calibration | $\widehat{p}_{T,c}(x)$ | Calibrated probability | Class-$c$ probability after temperature scaling |
| ECE | $B$ | Number of confidence bins | Total groups used by ECE |
| ECE | $B_m$ | Confidence bin $m$ | Cases falling in one confidence interval |
| ECE | $|B_m|$ | Bin size | Number of cases in bin $m$ |
| ECE | $\operatorname{acc}(B_m)$ | Bin accuracy | Correct share inside bin $m$ |
| ECE | $\operatorname{conf}(B_m)$ | Bin confidence | Mean confidence inside bin $m$ |
| Brier | $\widehat{p}_{i,c}$ | Predicted probability | Probability for case $i$ and class $c$ |
| Brier | $\mathbb{1}[y_i=c]$ | One-hot target | $1$ for the true class and $0$ otherwise |
| Policy | $\tau$ | Candidate review threshold | Minimum confidence for automatic action |
| Policy | $\mathcal{T}$ | Candidate threshold set | All thresholds considered |
| Policy | $A_i(\tau)$ | Acceptance indicator | Whether case $i$ meets threshold $\tau$ |
| Policy | $\operatorname{coverage}(\tau)$ | Automatic-decision share | Fraction accepted automatically |
| Policy | $\operatorname{review}(\tau)$ | Human-review share | Fraction deferred because of confidence |
| Policy | $\operatorname{SelAcc}(\tau)$ | Selective accuracy | Accuracy among accepted cases only |
| Policy | $\gamma_{\min}$ | Minimum coverage | Lowest allowed automatic-decision share |
| Policy | $a_{\min}$ | Minimum selective accuracy | Required accuracy among accepted cases |
| Policy | $\rho_{\max}$ | Maximum review rate | Largest permitted deferred share |
| Contract | $\ell_j$ | Lower bound for feature $j$ | Smallest allowed measurement |
| Contract | $u_j$ | Upper bound for feature $j$ | Largest allowed measurement |
| Contract | $\operatorname{valid}(x)$ | Input-contract result | Whether every required feature passes validation |
| Anomaly | $a(x)$ | Anomaly score | Familiarity score returned by the anomaly model |
| Anomaly | $\alpha$ | Tail probability | Calibration-tail fraction used as cutoff |
| Anomaly | $q_\alpha$ | Anomaly threshold | $\alpha$-quantile of calibration anomaly scores |
| Anomaly | $\operatorname{anomalous}(x)$ | Anomaly flag | Whether the score is below $q_\alpha$ |
| Routing | $\operatorname{route}(x)$ | Router output | Exactly one accept, review, or reject state |


## Session 3 equations carried into Session 4

These equations remain visible because a reliability policy must start from a validated and understood model.


### 1. Per-class F1 score

$$
F_{1,c}
=
\frac{2P_cR_c}{P_c+R_c}
=
\frac{2TP_c}{2TP_c+FP_c+FN_c}
$$

**Symbols**

- $P_c$ — precision for class $c$
- $R_c$ — recall for class $c$
- $TP_c$ — true positives for class $c$
- $FP_c$ — false positives for class $c$
- $FN_c$ — false negatives for class $c$
- $F_{1,c}$ — F1 score for class $c$

**Why it is used:** A class is considered usable only when both precision and recall are acceptable.

**Numerical example:** For `LOW`, let $TP=42$, $FP=6$, and $FN=8$. Then $F_1=84/(84+6+8)=0.857$.

**Interpretation boundary:** F1 measures predictive performance; deployment safety is not established by F1 alone.


### 2. Macro F1 score

$$
F_{1,\mathrm{macro}}
=
\frac{1}{C}
\sum_{c=1}^{C}
F_{1,c}
$$

**Symbols**

- $C$ — total number of classes
- $c$ — one class index
- $F_{1,c}$ — F1 score for class $c$
- $F_{1,\mathrm{macro}}$ — equal-weight average across classes

**Why it is used:** Rare classes are given the same voting power as common classes before the model is explained or controlled.

**Numerical example:** If class F1 values are $0.857$, $0.784$, and $0.765$, then Macro F1 is $(0.857+0.784+0.765)/3=0.802$.


### 3. Predicted class

$$
c^*(x)
=
\operatorname*{arg\,max}_{c\in\{1,\ldots,C\}}
\widehat{p}_c(x)
$$

**Symbols**

- $x$ — one input case
- $c$ — candidate class
- $C$ — number of classes
- $\widehat{p}_c(x)$ — probability assigned to class $c$
- $c^*(x)$ — class with the largest probability

**Why it is used:** The class actually selected by the model must be the class explained in Session 3 and routed in Session 4.

**Numerical example:** For probabilities $(0.1025,0.2521,0.6454)$, the maximum is $0.6454$, so $c^*(x)=\mathrm{HIGH}$.


### 4. Permutation importance

$$
I_j
=
\frac{1}{R}
\sum_{r=1}^{R}
\left(
S_0-S^{\mathrm{perm}}_{j,r}
\right)
$$

**Symbols**

- $I_j$ — importance assigned to feature $j$
- $R$ — number of shuffle repetitions
- $r$ — one repetition
- $S_0$ — baseline evaluation score
- $S^{\mathrm{perm}}_{j,r}$ — score after feature $j$ is shuffled in repetition $r$

**Why it is used:** Global model reliance is measured by observing how much the evaluation score falls when one feature is disrupted.

**Numerical example:** With $S_0=0.802$ and shuffled scores $0.700$, $0.720$, and $0.710$, $I_j=(0.102+0.082+0.092)/3=0.092$.

**Interpretation boundary:** Permutation importance is model- and sample-specific; causality is not established.


### 5. Local occlusion effect

$$
\Delta_{j,c^*}(x)
=
\widehat{p}_{c^*}(x)
-
\widehat{p}_{c^*}
\left(
x_{j\leftarrow b_j}
\right)
$$

**Symbols**

- $\Delta_{j,c^*}(x)$ — local effect of feature $j$ on the predicted class
- $c^*$ — predicted class
- $\widehat{p}_{c^*}(x)$ — original probability of the predicted class
- $b_j$ — reference value for feature $j$
- $x_{j\leftarrow b_j}$ — case after feature $j$ is replaced by $b_j$

**Why it is used:** One feature is replaced by a training reference so that one-case probability sensitivity can be inspected.

**Numerical example:** If $P(\mathrm{HIGH})$ falls from $0.645$ to $0.425$ after `drought_index` is replaced, then $\Delta=0.220$.

**Interpretation boundary:** This is a substitution sensitivity test, not a complete causal attribution.


### 6. Shapley credit allocation

$$
\phi_{j,c}
=
\sum_{S\subseteq F\setminus\{j\}}
\frac{|S|!\left(M-|S|-1\right)!}{M!}
\left[
v_c\left(S\cup\{j\}\right)-v_c(S)
\right]
$$

**Symbols**

- $\phi_{j,c}$ — credit assigned to feature $j$ for class $c$
- $F$ — set of all features
- $S$ — a coalition not containing feature $j$
- $M$ — number of features
- $|S|$ — number of features in coalition $S$
- $v_c(S)$ — class-$c$ output when coalition $S$ is known
- $S\cup\{j\}$ — coalition after feature $j$ is added

**Why it is used:** A feature's marginal model-output contribution is averaged across possible feature coalitions.

**Numerical example:** A large positive SHAP value means that the feature was assigned strong upward credit for the selected class score.

**Interpretation boundary:** SHAP remains inside the fitted model; real-world causality is not proven.


### 7. Additive SHAP reconstruction

$$
f_c(x)
=
\mathbb{E}\!\left[f_c(X)\right]
+
\sum_{j=1}^{M}
\phi_{j,c}
$$

**Symbols**

- $f_c(x)$ — class-$c$ model output for case $x$
- $\mathbb{E}[f_c(X)]$ — background baseline output
- $M$ — number of features
- $\phi_{j,c}$ — SHAP contribution of feature $j$

**Why it is used:** The baseline and all feature contributions are shown to reconstruct the explained model output.

**Numerical example:** If the baseline is $0.10$ and contributions are $0.22$, $0.12$, $0.08$, $-0.04$, and $0.01$, then the reconstructed score is $0.49$.

**Interpretation boundary:** The reconstructed output may be a raw score or log-odds rather than a probability.


### 8. Jaccard explanation stability

$$
J(A,B)
=
\frac{|A\cap B|}{|A\cup B|}
$$

**Symbols**

- $A$ — top-feature set from background A
- $B$ — top-feature set from background B
- $A\cap B$ — features shared by both sets
- $A\cup B$ — all unique features across both sets
- $J(A,B)$ — overlap score between $0$ and $1$

**Why it is used:** Top factors from two explanation backgrounds are compared before one ranking is trusted.

**Numerical example:** If four factors are shared and six are unique, then $J=4/6=0.667$.

**Interpretation boundary:** A moderate overlap supports cautious discussion, not a universal explanation.


## Session 4 equations: from confidence to controlled action

The selected and explained output is now passed through calibration, policy, validation, anomaly review, and deterministic routing.


### 9. Softmax probabilities

$$
\widehat{p}_c(x)
=
\frac{\exp\!\left(z_c(x)\right)}
{\sum_{k=1}^{C}\exp\!\left(z_k(x)\right)}
$$

**Symbols**

- $z_c(x)$ — logit for class $c$
- $C$ — number of classes
- $k$ — class index used in the denominator
- $\widehat{p}_c(x)$ — probability assigned to class $c$

**Why it is used:** Raw class scores are converted into a probability vector whose entries sum to one.

**Numerical example:** For logits $(0.20,1.10,2.04)$, softmax gives approximately $(0.1025,0.2521,0.6454)$.


### 10. Temperature selection

$$
T^*
=
\operatorname*{arg\,min}_{T>0}
\mathcal{L}_{\mathrm{NLL}}
\left(
y_{\mathrm{cal}},
\operatorname{softmax}
\left(
\frac{z_{\mathrm{cal}}}{T}
\right)
\right)
$$

**Symbols**

- $T$ — candidate positive temperature
- $T^*$ — temperature producing the smallest calibration NLL
- $\mathcal{L}_{\mathrm{NLL}}$ — negative log-likelihood
- $y_{\mathrm{cal}}$ — true calibration labels
- $z_{\mathrm{cal}}$ — logits on calibration cases

**Why it is used:** One scalar temperature is fitted on calibration data while the classifier parameters remain frozen.

**Numerical example:** In the running example, $T^*=1.4$ is assumed to be selected from the calibration period.

**Interpretation boundary:** Future test labels must not be used to fit the temperature.


### 11. Temperature-scaled probabilities

$$
\widehat{p}_{T,c}(x)
=
\operatorname{softmax}
\left(
\frac{z(x)}{T}
\right)_c
$$

**Symbols**

- $z(x)$ — vector of raw logits
- $T$ — selected temperature
- $\widehat{p}_{T,c}(x)$ — calibrated probability for class $c$
- $c$ — class being read from the calibrated vector

**Why it is used:** Probability sharpness is adjusted without retraining the classifier.

**Numerical example:** With logits $(0.20,1.10,2.04)$ and $T=1.4$, calibrated probabilities become approximately $(0.1510,0.2871,0.5619)$. `HIGH` remains selected, but confidence falls from $0.6454$ to $0.5619$.

**Interpretation boundary:** The predicted class ranking is preserved by positive temperature scaling, but calibration quality must still be measured.


### 12. Expected Calibration Error

$$
\operatorname{ECE}
=
\sum_{m=1}^{B}
\frac{|B_m|}{n}
\left|
\operatorname{acc}(B_m)
-
\operatorname{conf}(B_m)
\right|
$$

**Symbols**

- $B$ — number of confidence bins
- $B_m$ — cases assigned to bin $m$
- $|B_m|$ — number of cases in bin $m$
- $n$ — total number of cases
- $\operatorname{acc}(B_m)$ — observed accuracy in bin $m$
- $\operatorname{conf}(B_m)$ — mean confidence in bin $m$

**Why it is used:** Average confidence is compared with observed correctness across confidence regions.

**Numerical example:** For bin weights $0.20$, $0.45$, and $0.35$ with gaps $0.05$, $0.06$, and $0.05$, ECE is $0.20(0.05)+0.45(0.06)+0.35(0.05)=0.0545$.

**Interpretation boundary:** ECE depends on the binning scheme and should be reported with the number of bins.


### 13. Normalized multiclass Brier score

$$
\operatorname{Brier}
=
\frac{1}{nC}
\sum_{i=1}^{n}
\sum_{c=1}^{C}
\left(
\widehat{p}_{i,c}
-
\mathbb{1}[y_i=c]
\right)^2
$$

**Symbols**

- $n$ — number of cases
- $C$ — number of classes
- $\widehat{p}_{i,c}$ — predicted probability for case $i$, class $c$
- $y_i$ — true class of case $i$
- $\mathbb{1}[y_i=c]$ — one-hot target component

**Why it is used:** The complete probability vector is penalized against the true one-hot label; confident probability mass on wrong classes is punished.

**Numerical example:** For true class `HIGH` and probabilities $(0.1510,0.2871,0.5619)$, the one-case Brier score is $\left(0.1510^2+0.2871^2+(0.5619-1)^2\right)/3\approx0.0991$.

**Interpretation boundary:** Some libraries omit division by $C$. This notebook uses the normalized form displayed here.


### 14. Acceptance indicator and confidence

$$
A_i(\tau)
=
\mathbb{1}
\left[
\kappa(x_i)\geq\tau
\right],
\qquad
\kappa(x_i)
=
\max_c\widehat{p}_{T,c}(x_i)
$$

**Symbols**

- $\tau$ — candidate automatic-action threshold
- $\kappa(x_i)$ — maximum calibrated probability for case $i$
- $A_i(\tau)$ — indicator that case $i$ is accepted automatically
- $\mathbb{1}[\cdot]$ — indicator function

**Why it is used:** Every case is marked as automatically acceptable or confidence-deferred at a declared threshold.

**Numerical example:** For the running case, $\kappa=0.5619$. At $\tau=0.650$, $A_i(\tau)=0$, so automatic action is not permitted.


### 15. Coverage and review rate

$$
\operatorname{coverage}(\tau)
=
\frac{1}{n}
\sum_{i=1}^{n}
A_i(\tau),
\qquad
\operatorname{review}(\tau)
=
1-\operatorname{coverage}(\tau)
$$

**Symbols**

- $n$ — number of evaluated cases
- $A_i(\tau)$ — acceptance indicator for case $i$
- $\operatorname{coverage}(\tau)$ — automatic-decision fraction
- $\operatorname{review}(\tau)$ — confidence-deferred fraction

**Why it is used:** Operational workload is quantified rather than hidden.

**Numerical example:** If $70$ of $100$ calibration cases meet the threshold, coverage is $0.70$ and review rate is $0.30$.


### 16. Selective accuracy

$$
\operatorname{SelAcc}(\tau)
=
\frac{
\sum_{i=1}^{n}
A_i(\tau)
\mathbb{1}[\widehat{y}_i=y_i]
}{
\sum_{i=1}^{n}
A_i(\tau)
}
$$

**Symbols**

- $A_i(\tau)$ — acceptance indicator
- $\widehat{y}_i$ — predicted class
- $y_i$ — true class
- $\mathbb{1}[\widehat{y}_i=y_i]$ — indicator that case $i$ is correct
- $\operatorname{SelAcc}(\tau)$ — accuracy among accepted cases only

**Why it is used:** The quality of the automatically handled subset is measured separately from full-dataset accuracy.

**Numerical example:** If $59$ of $70$ accepted cases are correct, selective accuracy is $59/70=0.8429$.


### 17. Declared threshold policy

$$
\mathcal{F}
=
\left\{
\tau\in\mathcal{T}:
\operatorname{SelAcc}(\tau)\geq a_{\min},
\;
\operatorname{coverage}(\tau)\geq\gamma_{\min},
\;
\operatorname{review}(\tau)\leq\rho_{\max}
\right\}
$$

**Symbols**

- $\mathcal{T}$ — candidate threshold set
- $\mathcal{F}$ — feasible thresholds
- $a_{\min}$ — minimum selective accuracy
- $\gamma_{\min}$ — minimum coverage
- $\rho_{\max}$ — maximum review rate

**Why it is used:** The threshold is selected from an explicit operational policy rather than by intuition or future-label tuning.

**Numerical example:** With $a_{\min}=0.82$, $\gamma_{\min}=0.25$, and $\rho_{\max}=0.30$, a threshold producing coverage $0.70$, review $0.30$, and selective accuracy $0.8429$ is feasible.


### 18. Hard input contract

$$
\operatorname{valid}(x)
=
\bigwedge_{j=1}^{M}
\left(
\ell_j\leq x_j\leq u_j
\right)
$$

**Symbols**

- $M$ — number of monitored features
- $x_j$ — value of feature $j$
- $\ell_j$ — declared lower bound
- $u_j$ — declared upper bound
- $\bigwedge$ — logical AND across all feature checks
- $\operatorname{valid}(x)$ — result of the complete contract

**Why it is used:** Impossible measurements are blocked before feature engineering and `predict_proba`.

**Numerical example:** `temperature_c=34`, `rainfall_mm=12`, `humidity_pct=71`, and `soil_moisture=0.22` pass their ranges. `temperature_c=150` or `rainfall_mm=-5` fails the contract.

**Interpretation boundary:** A rejected request receives no class prediction.


### 19. Calibration-derived anomaly threshold

$$
q_\alpha
=
\operatorname{Quantile}_{\alpha}
\left(
\left\{
a(x_i):
x_i\in D_{\mathrm{cal}}
\right\}
\right)
$$

**Symbols**

- $a(x_i)$ — anomaly score of calibration case $i$
- $D_{\mathrm{cal}}$ — calibration dataset
- $\alpha$ — selected lower-tail fraction
- $q_\alpha$ — anomaly score cutoff

**Why it is used:** A familiarity boundary is derived from calibration behavior instead of being chosen from the future test set.

**Numerical example:** If the $2$nd percentile of calibration scores is $-0.050$, then $q_{0.02}=-0.050$.


### 20. Anomaly flag

$$
\operatorname{anomalous}(x)
=
\mathbb{1}
\left[
a(x)<q_\alpha
\right]
$$

**Symbols**

- $a(x)$ — anomaly score of the incoming valid request
- $q_\alpha$ — calibration-derived anomaly threshold
- $\operatorname{anomalous}(x)$ — binary anomaly-review flag

**Why it is used:** A syntactically valid request can still be deferred when its feature combination is historically unfamiliar.

**Numerical example:** For $q_\alpha=-0.050$, a case with score $0.120$ is familiar, while a valid case with score $-0.110$ is anomalous.

**Interpretation boundary:** An anomaly flag means unfamiliarity, not guaranteed model error.


### 21. Deterministic routing state

$$
\operatorname{route}(x)
=
\begin{cases}
\mathrm{REJECT\_INVALID\_INPUT},
& \neg\operatorname{valid}(x),\\[2mm]
\mathrm{HUMAN\_REVIEW\_ANOMALY},
& \operatorname{anomalous}(x),\\[2mm]
\mathrm{HUMAN\_REVIEW\_LOW\_CONFIDENCE},
& \kappa(x)<\tau,\\[2mm]
\mathrm{ACCEPT\_AI\_DECISION},
& \text{otherwise.}
\end{cases}
$$

**Symbols**

- $\operatorname{valid}(x)$ — input-contract result
- $\operatorname{anomalous}(x)$ — anomaly-review result
- $\kappa(x)$ — maximum calibrated class probability
- $\tau$ — selected review threshold
- $\operatorname{route}(x)$ — exactly one explicit system state

**Why it is used:** Priority is made deterministic and testable: invalid first, anomaly second, low confidence third, acceptance last.

**Numerical example:** The running case is valid and familiar, but $\kappa=0.5619<0.650$. Therefore, the route is `HUMAN_REVIEW_LOW_CONFIDENCE`.


## Integrated numerical case: every equation contributes to one decision

A single region-month case is followed from Session 3 model inspection into Session 4 operational control.

| Stage | Numerical evidence | Interpretation |
|---|---|---|
| Validation | Class F1 values: $0.857$, $0.784$, $0.765$; Macro F1 $=0.802$ | The selected model is legitimate enough to inspect, but safety has not yet been established. |
| Predicted output | Raw logits $(0.20,1.10,2.04)$ give probabilities $(0.1025,0.2521,0.6454)$ | `HIGH` is the class that must be explained and routed. |
| Global reliance | Shuffling `drought_index` lowers Macro F1 by an average of $0.092$ | The model relies globally on drought information. |
| Local occlusion | Replacing `drought_index` lowers $P(\mathrm{HIGH})$ from $0.645$ to $0.425$; $\Delta=0.220$ | The current drought value supports `HIGH` for this case. |
| SHAP | Baseline $0.10$ plus contributions $0.22+0.12+0.08-0.04+0.01$ gives class score $0.49$ | Contributions reconstruct the explained class score; causality is not claimed. |
| Stability | Four shared top features across six unique features gives $J=0.667$ | The explanation is stable enough for cautious review. |
| Temperature | $T=1.4$ changes probabilities to $(0.1510,0.2871,0.5619)$ | `HIGH` remains selected, but confidence becomes less sharp. |
| ECE evidence | Weighted confidence-accuracy gaps produce ECE $=0.0545$ | Confidence behavior is measured rather than trusted automatically. |
| Brier evidence | One-case normalized Brier $\approx0.0991$ | The full probability vector is compared with the true `HIGH` target. |
| Policy | $\tau=0.650$; calibration coverage $=0.70$; review $=0.30$; selective accuracy $=0.8429$ | The threshold satisfies the declared operating constraints. |
| Contract | All physical measurements are within declared bounds | The request is allowed to reach the model. |
| Anomaly | $a(x)=0.120$ and $q_{0.02}=-0.050$ | The request is familiar enough not to trigger anomaly review. |
| Final route | $0.5619<0.650$ | `HUMAN_REVIEW_LOW_CONFIDENCE` is returned. |

> **Key lesson:** a prediction can be valid, explainable, and non-anomalous while still being too uncertain for automatic action.


## How to read notebook outputs

Every code block in this notebook is followed by an output interpretation. The purpose is to make the evidence defensible, not just executable.

Use this rule while presenting: **first state what the output proves, then state what it does not prove.**


## Live output guide

| Output | What it proves | What not to claim |
|---|---|---|
| Calibration table | probability behavior was measured | calibration always improves performance |
| Reliability diagram | confidence can be compared to observed accuracy | confidence is ground truth |
| Threshold table | policy trade-off between coverage and accepted-case accuracy | threshold was tuned on future labels |
| Input contract | impossible inputs are refused before prediction | the model guessed safely |
| Anomaly review | valid but unfamiliar cases are deferred | anomaly means the label is wrong |
| Router assertions | all explicit states are reachable and testable | one example proves universal reliability |

## High-QA checklist for this notebook

- 2025 data must not be used to choose the temperature or review threshold.
- Raw and calibrated ECE/Brier must both be reported.
- The selected threshold must have a declared policy rule.
- Invalid measurements must return `REJECT_INVALID_INPUT` and `prediction = None`.
- Valid-but-unusual measurements must be eligible for `HUMAN_REVIEW_ANOMALY`.
- All four route assertions must pass: accept, low-confidence review, anomaly review, and invalid rejection.
- The final evidence record must include model version, policy version, threshold, status, and reason.

## Student deliverables

Submit the notebook or evidence screenshots showing:

1. selected temperature and raw/calibrated ECE + Brier;
2. threshold table and selected review threshold rule;
3. 2025 coverage, review rate, and selective accuracy at the frozen threshold;
4. one accepted case, one low-confidence review case, one anomaly-review case, and one invalid rejection case;
5. exported evidence JSON or a clear audit-record screenshot.

**Do not tune thresholds using the 2025 test labels.**

## Route priority

```
INVALID INPUT -> REJECT
VALID BUT ANOMALOUS -> HUMAN REVIEW
VALID + FAMILIAR BUT LOW CONFIDENCE -> HUMAN REVIEW
VALID + FAMILIAR + CONFIDENT -> ACCEPT
```


### Code block 1: purpose

**Before this code:** This cell locates the project root and adds the uploaded `src/` folder to `sys.path`. It ensures that the notebook uses the course modules already present in the package instead of relying on hidden local paths.

In [ ]:
import sys
import zipfile
from pathlib import Path

# The notebook may be opened from the extracted course package or beside
# Advanced_AI_Practical.zip. Both layouts are supported.
_current = Path().resolve()
_search_roots = [_current] + list(_current.parents)

PROJECT_ROOT = None
for _root in _search_roots:
    if (_root / "src").exists():
        PROJECT_ROOT = _root
        break

if PROJECT_ROOT is None:
    _zip_candidates = []
    for _root in _search_roots:
        _zip_candidates.append(_root / "Advanced_AI_Practical.zip")
    _zip_candidates.append(Path("/mnt/data/Advanced_AI_Practical.zip"))

    _course_zip = next((p for p in _zip_candidates if p.exists()), None)
    if _course_zip is None:
        raise FileNotFoundError(
            "The course package was not found. Place this notebook inside the "
            "extracted Advanced_AI_Practical folder or beside Advanced_AI_Practical.zip."
        )

    _extract_root = _course_zip.parent / "_Advanced_AI_Practical_extracted"
    _src_candidate = _extract_root / "src"
    if not _src_candidate.exists():
        _extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(_course_zip, "r") as _archive:
            _archive.extractall(_extract_root)

    if not _src_candidate.exists():
        raise FileNotFoundError(
            f"The archive was extracted, but src was not found at {_src_candidate}."
        )
    PROJECT_ROOT = _extract_root

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = SRC_DIR / "data"
REPORTS_DIR = SRC_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source dir:   {SRC_DIR}")
print(f"Data dir:     {DATA_DIR}")
print(f"Reports dir:  {REPORTS_DIR}")


### Output interpretation for code block 1

**After running it:** The printed paths confirm that `src/`, `src/data`, and `src/reports` were found. If these paths are wrong, the notebook should not continue.

### Code block 2: purpose

**Before this code:** This cell imports the scientific Python stack and the course modules needed for data loading, model comparison, reliability measurement, and routing.

In [ ]:
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from reliable_ai import (
    fuse_sources,
    compare_candidates,
    encode_target,
    build_feature_row,
    validate_measurements,
    MODEL_FEATURES,
    MONITOR_FEATURES,
    CLASS_NAMES,
)

np.set_printoptions(precision=4, suppress=True)
print("Imports ready")


### Output interpretation for code block 2

**After running it:** A successful import message means the notebook can access both public Python libraries and the uploaded course modules.

### Code block 3: purpose

**Before this code:** This cell loads the fused modeling table from `src/data/processed`. If the file is missing, the course data pipeline rebuilds it from the uploaded raw sources.

In [ ]:
fused_path = DATA_DIR / "processed" / "modeling_table.csv"
if fused_path.exists():
    df = pd.read_csv(fused_path)
    print(f"Loaded fused data: {df.shape}")
else:
    df = fuse_sources(DATA_DIR)
    print(f"Generated fused data: {df.shape}")

comparison = compare_candidates(df)
selected_model = comparison["selected_model"]
selected_name = comparison["selected_model_name"]
split = comparison["split"]
metrics = comparison["metrics"]
y_test = comparison["y_test"]

print(f"Selected model: {selected_name}")
print(metrics.to_string(index=False))


### Output interpretation for code block 3

**After running it:** The data shape and model comparison table show that the legitimate selected model is available. The leaky model may score higher, but it is not eligible for deployment.

### Code block 4: purpose

**Before this code:** This cell creates the chronological split: training years, the calibration year, and the untouched future test year. This prevents policy selection from leaking information from 2025.

In [ ]:
train = split["train"]
calibration = split["calibration"]
test = split["test"]

y_cal = encode_target(calibration["risk_label"])
X_cal = calibration[MODEL_FEATURES]
X_test = test[MODEL_FEATURES]

probs_cal_raw = selected_model.predict_proba(X_cal).astype(np.float64)
probs_test_raw = selected_model.predict_proba(X_test).astype(np.float64)

print("Chronological split")
print(f"  Train:       {train['year'].min()}-{train['year'].max()} | {len(train)} rows")
print(f"  Calibration: {calibration['year'].iloc[0]}      | {len(calibration)} rows")
print(f"  Test:        {test['year'].iloc[0]}      | {len(test)} rows")
print(f"  Probability matrices: calibration={probs_cal_raw.shape}, test={probs_test_raw.shape}")


### Output interpretation for code block 4

**After running it:** The row counts prove that calibration and test data are separate. The 2025 test set must remain untouched until final evaluation.

---
## 1. Temperature scaling

### Code block 5: purpose

**Before this code:** This cell implements temperature scaling. The selected classifier stays frozen; only the probability vector is softened or sharpened using the calibration year.

In [ ]:
def temperature_scale(probs: np.ndarray, T: float, eps: float = 1e-12) -> np.ndarray:
    clipped = np.clip(probs, eps, 1 - eps)
    logits = np.log(clipped) / T
    logits -= logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def temperature_log_loss(T: float, probs: np.ndarray, y: np.ndarray) -> float:
    return log_loss(y, temperature_scale(probs, T))

# TODO 1:
# Fit T on the 2024 calibration split only.
# Hint: use minimize_scalar(temperature_log_loss, args=(probs_cal_raw, y_cal), bounds=(0.1, 5.0), method="bounded")

result = None
T_OPT = None
probs_cal_calib = None
probs_test_calib = None

# YOUR CODE HERE

print(f"Optimal temperature T = {T_OPT:.4f}")


### Output interpretation for code block 5

**After running it:** The selected temperature is the calibration parameter. It should be recorded because every downstream confidence value depends on it.

---
## 2. Reliability metrics

### Code block 6: purpose

**Before this code:** This cell measures probability reliability using Brier score and Expected Calibration Error. These metrics evaluate confidence behavior, not only class correctness.

In [ ]:
def multiclass_brier(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_onehot = np.zeros_like(y_prob)
    y_onehot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean((y_prob - y_onehot) ** 2))


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    confidence = y_prob.max(axis=1)
    prediction = y_prob.argmax(axis=1)
    correct = (prediction == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (confidence > edges[i]) & (confidence <= edges[i + 1])
        if in_bin.any():
            ece += abs(correct[in_bin].mean() - confidence[in_bin].mean()) * in_bin.sum() / len(confidence)
    return float(ece)

# TODO 2:
# Build summary_df with raw and calibrated Brier/ECE for 2024 calibration and 2025 test.

summary_df = None

# YOUR CODE HERE

print(summary_df.round(6).to_string())


### Output interpretation for code block 6

**After running it:** Lower ECE or Brier is better. If a metric worsens, that result should still be reported because calibration is evidence, not marketing.

---
## 3. Threshold policy

### Code block 7: purpose

**Before this code:** This cell builds the threshold trade-off table and applies a declared review policy. Thresholds are selected from calibration data, not from the future test year.

In [ ]:
TARGET_SELECTIVE_ACCURACY = 0.82
MINIMUM_COVERAGE = 0.25
MAXIMUM_REVIEW_RATE = 0.30

# TODO 3:
# Build threshold_df for thresholds 0.50 to 0.95 step 0.025 using 2024 calibration probabilities.
# Columns: threshold, coverage, review_rate, selective_accuracy.
# Then select the feasible threshold with maximum coverage.

threshold_df = None
REVIEW_THRESHOLD = None

# YOUR CODE HERE

print(threshold_df.to_string(index=False))
print(f"Selected review threshold: {REVIEW_THRESHOLD:.3f}")


### Output interpretation for code block 7

**After running it:** The chosen threshold is justified by the declared policy on 2024. The threshold table shows the cost: higher thresholds usually reduce coverage and increase review load.

### Code block 8: purpose

**Before this code:** This cell evaluates the selected threshold on the untouched 2025 test year. This is a future check, not a second chance to tune the policy.

In [ ]:
# TODO 4:
# Evaluate the selected threshold on 2025 once. Do not tune using 2025.

test_conf = probs_test_calib.max(axis=1)
test_pred = probs_test_calib.argmax(axis=1)
accepted_2025 = test_conf >= REVIEW_THRESHOLD
coverage_2025 = accepted_2025.mean()
review_rate_2025 = 1.0 - coverage_2025
selective_acc_2025 = (test_pred[accepted_2025] == y_test[accepted_2025]).mean()

print(f"2025 coverage:           {coverage_2025:.4f}")
print(f"2025 review rate:        {review_rate_2025:.4f}")
print(f"2025 selective accuracy: {selective_acc_2025:.4f}")


### Output interpretation for code block 8

**After running it:** The 2025 numbers are the honest future behavior of the selected policy. If future selective accuracy is lower than calibration, that is drift evidence, not a reason to retune here.

---
## 4. Input contract

### Code block 9: purpose

**Before this code:** This cell defines hard input ranges and tests the input contract. Invalid measurements must be rejected before any call to `predict_proba`.

In [ ]:
VALID_REFERENCE_RECORD = {
    "month": 1,
    "temperature_c": 23.6,
    "rainfall_mm": 72.4,
    "humidity_pct": 68.5,
    "soil_moisture": 0.4599,
    "soil_ph": 6.48,
    "vegetation_index": 0.8958,
    "disease_cases": 41.3,
    "pest_reports": 28.8,
    "irrigation_capacity": 0.6734,
    "vulnerability_index": 0.352,
}

IMPOSSIBLE = {
    "month": 13,
    "temperature_c": 150.0,
    "rainfall_mm": -1.0,
    "humidity_pct": 500.0,
    "soil_moisture": 2.0,
    "soil_ph": 20.0,
    "vegetation_index": -1.0,
    "disease_cases": 9000.0,
    "pest_reports": 2000.0,
    "irrigation_capacity": 4.0,
    "vulnerability_index": -3.0,
}

print("Valid reference problems:", validate_measurements(VALID_REFERENCE_RECORD))
print("Impossible problems:", len(validate_measurements(IMPOSSIBLE)))


### Output interpretation for code block 9

**After running it:** The valid record should have no problems; the impossible record should list explicit violations. The model should never predict for the invalid request.

---
## 5. Anomaly review

### Code block 10: purpose

**Before this code:** This cell fits anomaly review on monitor features. It separates syntactically valid inputs from combinations that are valid but historically unusual.

In [ ]:
available_monitor = [f for f in MONITOR_FEATURES if f in train.columns]

anomaly_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("iforest", IsolationForest(n_estimators=250, contamination="auto", random_state=42, n_jobs=1)),
])

# TODO 5:
# Fit on train[available_monitor].
# Score the calibration split with decision_function.
# Select ANOMALY_THRESHOLD as the 2nd percentile of calibration scores.

ANOMALY_THRESHOLD = None

# YOUR CODE HERE

print(f"Anomaly threshold: {ANOMALY_THRESHOLD:.6f}")


### Output interpretation for code block 10

**After running it:** The anomaly threshold is a lower-tail cutoff from calibration scores. It is a review trigger, not a statement that the input is impossible.

---
## 6. Router

### Code block 11: purpose

**Before this code:** This cell implements the deterministic routing state machine. The order matters: invalid input, anomaly review, low-confidence review, then acceptance.

In [ ]:
def decision_router(record: dict, request_id: str = "demo") -> dict:
    # TODO 6:
    # 1. validate input; reject before prediction if problems exist.
    # 2. build feature_row using build_feature_row(record).
    # 3. compute anomaly score with anomaly_pipeline.decision_function.
    # 4. review anomaly if below threshold.
    # 5. predict probabilities, apply temperature scaling, compute confidence.
    # 6. review low confidence if confidence < REVIEW_THRESHOLD.
    # 7. otherwise accept.
    # Return a dictionary containing at least: request_id, status, reason, prediction, confidence, problems.

    raise NotImplementedError("Complete the router logic.")


### Output interpretation for code block 11

**After running it:** A ready message means the route order is defined. The router must return exactly one state with a reason and audit fields.

---
## 7. Four deterministic test cases

### Code block 12: purpose

**Before this code:** This cell defines hard input ranges and tests the input contract. Invalid measurements must be rejected before any call to `predict_proba`.

In [ ]:
ACCEPT_CASE = VALID_REFERENCE_RECORD

LOW_CONFIDENCE_CASE = {
    "month": 3,
    "temperature_c": 26.9,
    "rainfall_mm": 18.7,
    "humidity_pct": 60.9,
    "soil_moisture": 0.3002,
    "soil_ph": 5.71,
    "vegetation_index": 0.677,
    "disease_cases": 66.7,
    "pest_reports": 18.6,
    "irrigation_capacity": 0.6734,
    "vulnerability_index": 0.352,
}

ANOMALY_CASE = {
    "month": 10,
    "temperature_c": 12.6,
    "rainfall_mm": 76.9,
    "humidity_pct": 37.1,
    "soil_moisture": 0.6067,
    "soil_ph": 6.33,
    "vegetation_index": 0.4306,
    "disease_cases": 23.0,
    "pest_reports": 6.5,
    "irrigation_capacity": 0.6734,
    "vulnerability_index": 0.352,
}

cases = {
    "ACCEPT_CASE": ACCEPT_CASE,
    "LOW_CONFIDENCE_CASE": LOW_CONFIDENCE_CASE,
    "ANOMALY_CASE": ANOMALY_CASE,
    "IMPOSSIBLE": IMPOSSIBLE,
}

route_results = {name: decision_router(record, request_id=name) for name, record in cases.items()}
for name, result in route_results.items():
    print(name, "->", result["status"])


### Output interpretation for code block 12

**After running it:** The valid record should have no problems; the impossible record should list explicit violations. The model should never predict for the invalid request.

### Code block 13: purpose

**Before this code:** This cell runs four route tests. A reliable router must visibly produce ACCEPT, LOW-CONFIDENCE REVIEW, ANOMALY REVIEW, and REJECT.

In [ ]:
expected_status = {
    "ACCEPT_CASE": "ACCEPT_AI_DECISION",
    "LOW_CONFIDENCE_CASE": "HUMAN_REVIEW_LOW_CONFIDENCE",
    "ANOMALY_CASE": "HUMAN_REVIEW_ANOMALY",
    "IMPOSSIBLE": "REJECT_INVALID_INPUT",
}

for name, expected in expected_status.items():
    actual = route_results[name]["status"]
    assert actual == expected, f"{name}: expected {expected}, got {actual}"
    print(f"[PASS] {name} -> {actual}")

assert route_results["IMPOSSIBLE"]["prediction"] is None
print("[PASS] Impossible request has no prediction.")


### Output interpretation for code block 13

**After running it:** Each printed route should have one status, one reason, and no hidden second decision. Accepted cases may include explanation evidence; rejected invalid cases should have no prediction.

---
## Evidence questions

1. Did temperature scaling improve 2025 ECE on this run?
2. Which review threshold was selected, and what rule selected it?
3. What percentage of 2025 cases were automatically accepted?
4. Which guardrail blocks impossible input before the model?
5. Why can valid input still require human review?


### Code block 14: purpose

**Before this code:** This code block performs the next operational step in the reliability pipeline. Read its output as evidence for routing, not as a standalone score.

In [ ]:
# TODO 8:
# Export threshold_df and a short evidence dictionary to REPORTS_DIR.
# Recommended output names:
# - session4_threshold_analysis_student.csv
# - session4_evidence_record_student.json

print("Lab complete after evidence export.")


### Output interpretation for code block 14

**After running it:** The output should be interpreted against the stated policy. A number is useful only when it changes the reliability decision or documents evidence.